In [ ]:
import os

# Ensure we're always running from the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())
print(os.listdir("."))

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set these two values to match the run you want to explore.
# RUN_ID must match the run_id used when the pipeline was executed.
# SAMPLE  must match the sample ID (FASTQ filename prefix).

RUN_ID = "SRR1258218_chr22"
SAMPLE = "SRR1258218"

# Derived paths — no need to change these
BAM     = f"results/{RUN_ID}/bam/{SAMPLE}.sorted.bam"
RAW_VCF = f"results/{RUN_ID}/vcf/{SAMPLE}.raw.vcf"
FLT_VCF = f"results/{RUN_ID}/vcf/{SAMPLE}.filtered.vcf"

print(f"Run:    {RUN_ID}")
print(f"Sample: {SAMPLE}")
print(f"BAM:    {BAM}  (exists: {os.path.exists(BAM)})")
print(f"VCF:    {FLT_VCF}  (exists: {os.path.exists(FLT_VCF)})")

In [ ]:
import subprocess

# 1. How many reads actually aligned?
result = subprocess.run(
    ["samtools", "flagstat", BAM],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# 2. Peek at the raw VCF — header + first variant
with open(RAW_VCF) as f:
    for line in f:
        print(line.strip())
        if not line.startswith("#"):
            break

In [ ]:
# 3. Count variants at each stage
def count_variants(vcf_path):
    result = subprocess.run(
        f"grep -v '^#' {vcf_path} | wc -l",
        shell=True, capture_output=True, text=True
    )
    return int(result.stdout.strip())

raw      = count_variants(RAW_VCF)
filtered = count_variants(FLT_VCF)

print(f"Raw variants:      {raw}")
print(f"Filtered variants: {filtered}")
print(f"Removed by filter: {raw - filtered}")

In [ ]:
# 4. Look at the actual variant lines
with open(FLT_VCF) as f:
    for line in f:
        if not line.startswith("#"):
            fields = line.strip().split("\t")
            print(f"Chr: {fields[0]}  Pos: {fields[1]}  "
                  f"Ref: {fields[3]}  Alt: {fields[4]}  "
                  f"Qual: {fields[5]}  Filter: {fields[6]}")

In [ ]:
# 5. How many reads aligned at all?
result = subprocess.run(["samtools", "flagstat", BAM], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# 6. How deep is the coverage at any position?
result = subprocess.run(
    f"samtools depth {BAM} | sort -k3 -rn | head -10",
    shell=True, capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# 7. Does the BAM actually have reads in it?
result = subprocess.run(["samtools", "view", "-c", BAM], capture_output=True, text=True)
print(result.stdout)